In [ ]:
# ==========================================
# BẮT BUỘC DÙNG TF_KERAS ĐỂ TRÁNH LỖI "3.0" CỦA TENSORFLOW MỚI
# ==========================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import tf_keras # <--- KHÔNG ĐƯỢC ĐỔI THÀNH TENSORFLOW
import numpy as np
import base64
from io import BytesIO
from PIL import Image
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import requests
import urllib3
from playwright.async_api import async_playwright
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import gc 
from starlette.concurrency import run_in_threadpool 

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = "nhatphivo781@gmail.com" 
SENDER_PASSWORD = "oavu cbps vvga npcg"

nest_asyncio.apply()
app = FastAPI()

app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)

AVAILABLE_MODELS = {
    "EfficientNet": "models/efficientnet_defacement_model.h5",
    "ResNet50": "models/ResNet50_defacement_model.h5",    
    "MobileNet_BaseLine": "models/mobileNetv3_baseline_defacement_model.h5",
    "MobileNet_Finetuning": "models/mobileNetv3_finetuning_defacement_model.h5"
}

loaded_model_data = {"name": None, "model": None}

def get_ai_model(model_name):
    if model_name not in AVAILABLE_MODELS:
        raise ValueError(f"Model '{model_name}' chưa được khai báo!")
    
    if loaded_model_data["name"] != model_name:
        print(f"\n🧹 Đang dọn dẹp RAM để nhường chỗ cho {model_name}...")
        loaded_model_data["model"] = None 
        tf_keras.backend.clear_session() 
        gc.collect()      
        
        print(f"⏳ Đang nạp model {model_name}...")
        
        # Ánh xạ hàm Hard Swish về Swish chuẩn để an toàn tuyệt đối
        safe_objects = {
            'hard_swish': tf_keras.activations.swish,
            'swish': tf_keras.activations.swish
        }
        
        loaded_model_data["model"] = tf_keras.models.load_model(
            AVAILABLE_MODELS[model_name], 
            compile=False,
            custom_objects=safe_objects
        )
        loaded_model_data["name"] = model_name
        print(f"✅ Đã tải xong {model_name}!")
        
    return loaded_model_data["model"]

class DetectionRequest(BaseModel):
    type: str; data: str; model_name: str; email: str

def send_email_alert(target_email, source, confidence):
    try:
        msg = MIMEMultipart()
        msg['From'] = SENDER_EMAIL; msg['To'] = target_email
        msg['Subject'] = f"🚨 CẢNH BÁO BẢO MẬT: Phát hiện tấn công Deface!"
        body = f"Nguồn: {source}\nMức độ nguy hiểm: {confidence}%\nVui lòng kiểm tra ngay!"
        msg.attach(MIMEText(body, 'plain', 'utf-8'))
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls(); server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, target_email, msg.as_string()); server.quit()
    except Exception as e: pass

def process_image(img):
    img = img.convert('RGB')
    img_resized = img.resize((224, 224))
    img_array = np.array(img_resized, dtype=np.float32)
    return np.expand_dims(img_array, axis=0)

def process_base64_image(base64_str):
    return process_image(Image.open(BytesIO(base64.b64decode(base64_str.split(',')[1]))))

def process_bytes_image(image_bytes):
    return process_image(Image.open(BytesIO(image_bytes)))

async def capture_screenshot_from_url(url: str):
    if not url.startswith("http://") and not url.startswith("https://"): url = "http://" + url
    try:
        resp = requests.get(url, verify=False, timeout=10)
        if 'image' in resp.headers.get('Content-Type', '').lower(): return resp.content
    except: pass 
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            page = await (await browser.new_context(ignore_https_errors=True)).new_page()
            await page.set_viewport_size({"width": 1366, "height": 768})
            try: await page.goto(url, wait_until="load", timeout=30000)
            except: pass 
            await page.wait_for_timeout(2000)
            screenshot_bytes = await page.screenshot()
            await browser.close()
            return screenshot_bytes
    except: return None

@app.post("/detect")
async def detect_deface(request: DetectionRequest):
    try:
        source_name = "Ảnh chụp trực tiếp"
        try: current_model = await run_in_threadpool(get_ai_model, request.model_name)
        except ValueError as ve: return {"error": str(ve)}
            
        if request.type == 'image': img_tensor = process_base64_image(request.data)
        elif request.type == 'url':
            source_name = request.data
            screenshot_bytes = await capture_screenshot_from_url(source_name)
            if not screenshot_bytes: return {"error": "Lỗi kết nối hoặc trang tải quá chậm!"}
            img_tensor = process_bytes_image(screenshot_bytes)
        else: return {"error": "Loại request không hợp lệ!"}

        prediction = await run_in_threadpool(current_model.predict, img_tensor, verbose=0)
        deface_prob = float(prediction[0][1]) 
        is_defaced = bool(deface_prob > 0.65)
        conf = round(deface_prob * 100 if is_defaced else (1 - deface_prob) * 100, 2)
        
        if is_defaced and request.email:
            await run_in_threadpool(send_email_alert, request.email, source_name, conf)
        
        return {"is_defaced": is_defaced, "confidence": conf, "message": "Hoàn tất"}
    except Exception as e: return {"error": f"Lỗi server: {str(e)}"}

if __name__ == "__main__":
    print("\n" + "="*50)
    print("🚀 ĐANG KHỞI ĐỘNG BACKEND SERVER TẠI HTTP://127.0.0.1:8000")
    print("⏹️ Nhấn nút 'Interrupt' (hoặc bấm 0, 0) nếu muốn tắt server.")
    print("="*50 + "\n")
    
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    await server.serve()


🚀 ĐANG KHỞI ĐỘNG BACKEND SERVER TẠI HTTP://127.0.0.1:8000
⏹️ Nhấn nút 'Interrupt' (hoặc bấm 0, 0) nếu muốn tắt server.


🧹 Đang dọn dẹp RAM để nhường chỗ cho MobileNet_BaseLine...

⏳ Đang nạp model MobileNet_BaseLine...


In [2]:
import h5py
import json
import os

# Đảm bảo đây là đường dẫn đúng trên máy bạn
models = [
    r"D:\HocTap\IE105\Defaced detection\DefaceDetector_Backend\models\efficientnet_defacement_model.h5",
    r"D:\HocTap\IE105\Defaced detection\DefaceDetector_Backend\models\ResNet50_defacement_model.h5",
    r"D:\HocTap\IE105\Defaced detection\DefaceDetector_Backend\models\mobileNetv3_baseline_defacement_model.h5",
    r"D:\HocTap\IE105\Defaced detection\DefaceDetector_Backend\models\mobileNetv3_finetuning_defacement_model.h5"
]

def master_cleanser(node):
    if isinstance(node, dict):
        # 1. Trị bệnh DTypePolicy
        if node.get('class_name') == 'DTypePolicy':
            return node.get('config', {}).get('name', 'float32')
        
        # 2. Trị bệnh dư thừa tham số 'groups' (Gây lỗi DepthwiseConv2D)
        if 'groups' in node:
            del node['groups']
            
        # 3. Trị bệnh batch_shape
        if 'batch_shape' in node:
            node['batch_input_shape'] = node.pop('batch_shape')

        for k, v in list(node.items()):
            node[k] = master_cleanser(v)
        return node
    
    elif isinstance(node, list):
        return [master_cleanser(x) for x in node]
    
    elif isinstance(node, str):
        # 4. Trị bệnh hàm kích hoạt rác
        if node == 'hard_silu':
            return 'hard_swish'
        return node
    else:
        return node

print("💉 ĐANG TIẾN HÀNH TẨY RỬA CẤU TRÚC FILE .H5...\n" + "="*50)

for path in models:
    if not os.path.exists(path):
        continue
    try:
        with h5py.File(path, 'r+') as f:
            if 'model_config' in f.attrs:
                config_str = f.attrs['model_config']
                if isinstance(config_str, bytes):
                    config_str = config_str.decode('utf-8')
                
                # Giải mã -> Tẩy rửa -> Nén lại
                config_dict = json.loads(config_str)
                healed_dict = master_cleanser(config_dict)
                new_config_str = json.dumps(healed_dict)
                
                f.attrs['model_config'] = new_config_str.encode('utf-8')
                print(f"✅ Làm sạch thành công: {os.path.basename(path)}")
    except Exception as e:
        print(f"❌ Lỗi: {e}")

print("="*50 + "\n🎉 TẤT CẢ FILE ĐÃ ĐẠT CHUẨN KERAS 2!")

💉 ĐANG TIẾN HÀNH TẨY RỬA CẤU TRÚC FILE .H5...
✅ Làm sạch thành công: efficientnet_defacement_model.h5
✅ Làm sạch thành công: mobileNetv3_baseline_defacement_model.h5
🎉 TẤT CẢ FILE ĐÃ ĐẠT CHUẨN KERAS 2!
